# AtmoSync
## 02 - Data Cleaning and Feature Engineering

### Objective

The purpose of this notebook is to clean the raw dataset and create meaningful analytical features.

### Cleaning Tasks

1. Convert timestamp to datetime.
2. Convert NDVI to numeric.
3. Identify invalid values.
4. Check missing values.
5. Check duplicate records.
6. Validate environmental ranges.
7. Create time-based features.
8. Create environmental indicators.
9. Export the processed dataset.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "raw"
    / "Microclimate_dataset.csv"
)

PROCESSED_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / "microclimate_cleaned.csv"
)

df = pd.read_csv(RAW_PATH)

print("Raw dataset shape:", df.shape)

Raw dataset shape: (8640, 12)


In [3]:
# Convert timestamp from text to datetime

df["cdTimestamp"] = pd.to_datetime(
    df["cdTimestamp"],
    errors="coerce"
)

print(df["cdTimestamp"].dtype)

datetime64[us]


In [4]:
# Convert NDVI to numeric.
# Invalid values will become NaN.

df["NDVI"] = pd.to_numeric(
    df["NDVI"],
    errors="coerce"
)

print("Missing NDVI after conversion:")
print(df["NDVI"].isna().sum())

Missing NDVI after conversion:
1


In [5]:
# Sort by location and timestamp

df = df.sort_values(
    ["Location_ID", "cdTimestamp"]
)

# Interpolate missing NDVI values within each location

df["NDVI"] = (
    df.groupby("Location_ID")["NDVI"]
    .transform(
        lambda x: x.interpolate(
            limit_direction="both"
        )
    )
)

In [6]:
# Extract hour from timestamp

df["Hour"] = (
    df["cdTimestamp"].dt.hour
    + df["cdTimestamp"].dt.minute / 60
)

# Extract minute

df["Minute"] = df["cdTimestamp"].dt.minute

In [7]:
def classify_time_period(hour):

    if 0 <= hour < 6:
        return "Night"

    elif 6 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["Time_Period"] = df["Hour"].apply(
    classify_time_period
)

In [8]:
# Calculate the average temperature of each location

location_temperature_mean = (
    df.groupby("Location_ID")["Temperature"]
    .transform("mean")
)

# Difference between the current temperature
# and the normal average temperature of that location.

df["Temperature_Anomaly"] = (
    df["Temperature"]
    - location_temperature_mean
)   

Positive anomaly → warmer than the location's average
Negative anomaly → cooler than the location's average
Zero             → approximately average

In [9]:
# Magnus approximation for dew point

a = 17.27
b = 237.7

gamma = (
    (a * df["Temperature"])
    / (b + df["Temperature"])
    + np.log(df["Humidity"] / 100)
)

df["Dew_Point_C"] = (
    b * gamma
    / (a - gamma)
)

In [10]:
df["NDVI_Category"] = pd.cut(
    df["NDVI"],
    bins=[-1, 0.2, 0.5, 0.7, 1],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

In [11]:
df["Solar_Radiation_Band"] = pd.cut(
    df["Solar_Radiation"],
    bins=[
        -np.inf,
        100,
        300,
        600,
        np.inf
    ],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

In [12]:
# Save the cleaned dataset

df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("Cleaned dataset saved successfully.")
print(PROCESSED_PATH)

Cleaned dataset saved successfully.
c:\PlayGround\Infotact Data Analytics\AtmoSync\dataset\processed\microclimate_cleaned.csv
